In [0]:
CATALOG = "adwm_wh"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dimemployee"

source_tables = [
    f"{CATALOG}.{SILVER_SCHEMA}.employee",
    f"{CATALOG}.{SILVER_SCHEMA}.person",
    f"{CATALOG}.{SILVER_SCHEMA}.employeedepartmenthistory",
    f"{CATALOG}.{SILVER_SCHEMA}.department",
    f"{CATALOG}.{SILVER_SCHEMA}.shift"
]

print(f"Target table: {TARGET_TABLE}")
print("Source tables:")
for table_name in source_tables:
    print(f"  - {table_name}")

In [0]:
%sql
WITH source_prepared AS (
    SELECT
        emp.BusinessEntityID,
        sha2(
            concat_ws(
                '||',
                coalesce(cast(emp.BusinessEntityID AS STRING), '∅'),
                coalesce(cast(emp.NationalIDNumber AS STRING), '∅'),
                coalesce(cast(emp.LoginID AS STRING), '∅')
            ),
            256
        ) AS EmployeeVariantHash,
        emp.NationalIDNumber,
        emp.LoginID,
        emp.JobTitle,
        emp.BirthDate,
        emp.Gender,
        emp.MaritalStatus,
        emp.HireDate,
        emp.SalariedFlag,
        emp.VacationHours,
        emp.SickLeaveHours,
        emp.CurrentFlag,
        hist.DepartmentID,
        dept.Name AS DepartmentName,
        dept.GroupName,
        hist.ShiftID,
        shf.Name AS ShiftName,
        hist.StartDate AS DepartmentStartDate,
        hist.EndDate AS DepartmentEndDate,
        per.FirstName,
        per.MiddleName,
        per.LastName,
        trim(concat_ws(' ', per.FirstName, per.MiddleName, per.LastName)) AS FullName,
        per.Title,
        per.Suffix,
        per.EmailPromotion,
        per.Demographics,
        emp.rowguid AS RowGUID
    FROM adwm_wh.silver.employee emp
    LEFT JOIN adwm_wh.silver.person per
        ON emp.BusinessEntityID = per.BusinessEntityID
       AND (
            coalesce(per.FirstName, 'UNKNOWN') <> 'UNKNOWN'
            OR coalesce(per.LastName, 'UNKNOWN') <> 'UNKNOWN'
            OR coalesce(per.rowguid, 'UNKNOWN') <> 'UNKNOWN'
       )
    LEFT JOIN adwm_wh.silver.employeedepartmenthistory hist
        ON emp.BusinessEntityID = hist.BusinessEntityID
       AND hist.EndDate IS NULL
    LEFT JOIN adwm_wh.silver.department dept
        ON hist.DepartmentID = dept.DepartmentID
    LEFT JOIN adwm_wh.silver.shift shf
        ON hist.ShiftID = shf.ShiftID
),
source_data AS (
    SELECT
        BusinessEntityID,
        EmployeeVariantHash,
        NationalIDNumber,
        LoginID,
        JobTitle,
        BirthDate,
        Gender,
        MaritalStatus,
        HireDate,
        SalariedFlag,
        VacationHours,
        SickLeaveHours,
        CurrentFlag,
        DepartmentID,
        DepartmentName,
        GroupName,
        ShiftID,
        ShiftName,
        DepartmentStartDate,
        DepartmentEndDate,
        FirstName,
        MiddleName,
        LastName,
        FullName,
        Title,
        Suffix,
        EmailPromotion,
        Demographics,
        RowGUID,
        current_timestamp() AS ModifiedDate,
        sha2(
            concat_ws(
                '||',
                coalesce(cast(BusinessEntityID AS STRING), '∅'),
                coalesce(cast(EmployeeVariantHash AS STRING), '∅'),
                coalesce(cast(NationalIDNumber AS STRING), '∅'),
                coalesce(cast(LoginID AS STRING), '∅'),
                coalesce(cast(JobTitle AS STRING), '∅'),
                coalesce(cast(BirthDate AS STRING), '∅'),
                coalesce(cast(Gender AS STRING), '∅'),
                coalesce(cast(MaritalStatus AS STRING), '∅'),
                coalesce(cast(HireDate AS STRING), '∅'),
                coalesce(cast(SalariedFlag AS STRING), '∅'),
                coalesce(cast(VacationHours AS STRING), '∅'),
                coalesce(cast(SickLeaveHours AS STRING), '∅'),
                coalesce(cast(CurrentFlag AS STRING), '∅'),
                coalesce(cast(DepartmentID AS STRING), '∅'),
                coalesce(cast(DepartmentName AS STRING), '∅'),
                coalesce(cast(GroupName AS STRING), '∅'),
                coalesce(cast(ShiftID AS STRING), '∅'),
                coalesce(cast(ShiftName AS STRING), '∅'),
                coalesce(cast(DepartmentStartDate AS STRING), '∅'),
                coalesce(cast(DepartmentEndDate AS STRING), '∅'),
                coalesce(cast(FirstName AS STRING), '∅'),
                coalesce(cast(MiddleName AS STRING), '∅'),
                coalesce(cast(LastName AS STRING), '∅'),
                coalesce(cast(FullName AS STRING), '∅'),
                coalesce(cast(Title AS STRING), '∅'),
                coalesce(cast(Suffix AS STRING), '∅'),
                coalesce(cast(EmailPromotion AS STRING), '∅'),
                coalesce(cast(Demographics AS STRING), '∅'),
                coalesce(cast(RowGUID AS STRING), '∅')
            ),
            256
        ) AS ChangeHash
    FROM source_prepared
)
MERGE INTO adwm_wh.gold.dimemployee AS target
USING source_data AS source
ON target.EmployeeVariantHash = source.EmployeeVariantHash
WHEN MATCHED AND source.ChangeHash <> sha2(
    concat_ws(
        '||',
        coalesce(cast(target.BusinessEntityID AS STRING), '∅'),
        coalesce(cast(target.EmployeeVariantHash AS STRING), '∅'),
        coalesce(cast(target.NationalIDNumber AS STRING), '∅'),
        coalesce(cast(target.LoginID AS STRING), '∅'),
        coalesce(cast(target.JobTitle AS STRING), '∅'),
        coalesce(cast(target.BirthDate AS STRING), '∅'),
        coalesce(cast(target.Gender AS STRING), '∅'),
        coalesce(cast(target.MaritalStatus AS STRING), '∅'),
        coalesce(cast(target.HireDate AS STRING), '∅'),
        coalesce(cast(target.SalariedFlag AS STRING), '∅'),
        coalesce(cast(target.VacationHours AS STRING), '∅'),
        coalesce(cast(target.SickLeaveHours AS STRING), '∅'),
        coalesce(cast(target.CurrentFlag AS STRING), '∅'),
        coalesce(cast(target.DepartmentID AS STRING), '∅'),
        coalesce(cast(target.DepartmentName AS STRING), '∅'),
        coalesce(cast(target.GroupName AS STRING), '∅'),
        coalesce(cast(target.ShiftID AS STRING), '∅'),
        coalesce(cast(target.ShiftName AS STRING), '∅'),
        coalesce(cast(target.DepartmentStartDate AS STRING), '∅'),
        coalesce(cast(target.DepartmentEndDate AS STRING), '∅'),
        coalesce(cast(target.FirstName AS STRING), '∅'),
        coalesce(cast(target.MiddleName AS STRING), '∅'),
        coalesce(cast(target.LastName AS STRING), '∅'),
        coalesce(cast(target.FullName AS STRING), '∅'),
        coalesce(cast(target.Title AS STRING), '∅'),
        coalesce(cast(target.Suffix AS STRING), '∅'),
        coalesce(cast(target.EmailPromotion AS STRING), '∅'),
        coalesce(cast(target.Demographics AS STRING), '∅'),
        coalesce(cast(target.RowGUID AS STRING), '∅')
    ),
    256
) THEN UPDATE SET
    target.BusinessEntityID = source.BusinessEntityID,
    target.NationalIDNumber = source.NationalIDNumber,
    target.LoginID = source.LoginID,
    target.JobTitle = source.JobTitle,
    target.BirthDate = source.BirthDate,
    target.Gender = source.Gender,
    target.MaritalStatus = source.MaritalStatus,
    target.HireDate = source.HireDate,
    target.SalariedFlag = source.SalariedFlag,
    target.VacationHours = source.VacationHours,
    target.SickLeaveHours = source.SickLeaveHours,
    target.CurrentFlag = source.CurrentFlag,
    target.DepartmentID = source.DepartmentID,
    target.DepartmentName = source.DepartmentName,
    target.GroupName = source.GroupName,
    target.ShiftID = source.ShiftID,
    target.ShiftName = source.ShiftName,
    target.DepartmentStartDate = source.DepartmentStartDate,
    target.DepartmentEndDate = source.DepartmentEndDate,
    target.FirstName = source.FirstName,
    target.MiddleName = source.MiddleName,
    target.LastName = source.LastName,
    target.FullName = source.FullName,
    target.Title = source.Title,
    target.Suffix = source.Suffix,
    target.EmailPromotion = source.EmailPromotion,
    target.Demographics = source.Demographics,
    target.RowGUID = source.RowGUID,
    target.ModifiedDate = source.ModifiedDate
WHEN NOT MATCHED THEN INSERT (
    BusinessEntityID,
    EmployeeVariantHash,
    NationalIDNumber,
    LoginID,
    JobTitle,
    BirthDate,
    Gender,
    MaritalStatus,
    HireDate,
    SalariedFlag,
    VacationHours,
    SickLeaveHours,
    CurrentFlag,
    DepartmentID,
    DepartmentName,
    GroupName,
    ShiftID,
    ShiftName,
    DepartmentStartDate,
    DepartmentEndDate,
    FirstName,
    MiddleName,
    LastName,
    FullName,
    Title,
    Suffix,
    EmailPromotion,
    Demographics,
    RowGUID,
    ModifiedDate
) VALUES (
    source.BusinessEntityID,
    source.EmployeeVariantHash,
    source.NationalIDNumber,
    source.LoginID,
    source.JobTitle,
    source.BirthDate,
    source.Gender,
    source.MaritalStatus,
    source.HireDate,
    source.SalariedFlag,
    source.VacationHours,
    source.SickLeaveHours,
    source.CurrentFlag,
    source.DepartmentID,
    source.DepartmentName,
    source.GroupName,
    source.ShiftID,
    source.ShiftName,
    source.DepartmentStartDate,
    source.DepartmentEndDate,
    source.FirstName,
    source.MiddleName,
    source.LastName,
    source.FullName,
    source.Title,
    source.Suffix,
    source.EmailPromotion,
    source.Demographics,
    source.RowGUID,
    source.ModifiedDate
);

In [0]:
%sql
SELECT
    COUNT(*) AS dimemployee_count,
    COUNT(DISTINCT EmployeeVariantHash) AS distinct_employeevarianthashes,
    COUNT(DISTINCT BusinessEntityID) AS distinct_businessentityids
FROM adwm_wh.gold.dimemployee;

In [0]:
%sql
SELECT
    EmployeeKey,
    BusinessEntityID,
    EmployeeVariantHash,
    NationalIDNumber,
    LoginID,
    FirstName,
    LastName,
    DepartmentID,
    ShiftID
FROM adwm_wh.gold.dimemployee
ORDER BY BusinessEntityID, NationalIDNumber, LoginID
LIMIT 25;